In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import io
import requests

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 37
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## Political Finance Pipeline (IDEA)

**Source:** International IDEA Political Finance Database
**Access:** Automated — direct .xlsx export endpoint
**Download instructions:** See `docs/instructions_data_maintenance.md` — TI_POLFINANCE section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Money-in-politics regulation (bans/limits, public funding, spending, oversight) | Political finance integrity / anti-corruption | Primary tier 2 |

### Note
Source ID retained as TI_POLFINANCE for continuity, but the structured data is from
International IDEA's Political Finance Database (the authoritative structured source);
Transparency International publishes analysis/standards, not a comparable country panel.

### Honest scope
- **De jure regulation only** — codes whether rules exist on paper, NOT enforcement or
  compliance (IDEA explicitly notes laws on the books ≠ adherence).
- **National-level** regulations.
- **Wave-updated**, not annual — current cross-section reflecting periodic update rounds.
- 181 countries; ~58 questions across four categories.

In [2]:
# Download the IDEA Political Finance Database as .xlsx via the direct export endpoint.
# themeId=302 = Political Finance Database; world=all = all countries. No hardcoded dates.
POLFIN_EXPORT_URL = "https://www.idea.int/data-tools/export?type=region_only&themeId=302&world=all&loc=home"

r = requests.get(POLFIN_EXPORT_URL, headers=BROWSER_HEADERS, timeout=60)
ctype = r.headers.get('Content-Type', '')
if r.status_code != 200 or 'spreadsheet' not in ctype.lower():
    raise RuntimeError(f"Unexpected response: {r.status_code}, {ctype}")
print(f"Downloaded {len(r.content)/1024:.1f}KB, type OK")

# Inspect sheet structure
xl = pd.ExcelFile(io.BytesIO(r.content), engine='openpyxl')
print(f"Sheets: {xl.sheet_names}")
for sheet in xl.sheet_names[:3]:
    df = xl.parse(sheet, header=None, nrows=5)
    print(f"\n=== {sheet} === preview:")
    print(df.to_string())

Downloaded 127.5KB, type OK
Sheets: ['All', 'Bans and limits on p...', 'Public funding', 'Regulations of spending', 'Reporting, oversight...']

=== All === preview:
            0     1     2                                                                                                                3                                                                                                         4                                                                                                   5                                                                                            6                                                                                                           7                                                                                                    8                                                                                                   9                                                                                         

In [3]:
# Load the 'All' sheet (header row 0 = question labels) for processing.
r = requests.get(POLFIN_EXPORT_URL, headers=BROWSER_HEADERS, timeout=60)
raw = pd.read_excel(io.BytesIO(r.content), sheet_name='All', engine='openpyxl', header=0)
print(f"Raw: {raw.shape}")

# --- Country key columns vs question columns ---
key_cols = ['Country', 'ISO2', 'ISO3']
question_cols = [c for c in raw.columns if c not in key_cols]

# --- Auto-detect BINARY questions: columns whose non-null answers are dominated by the fixed
#     binary/categorical answer set (excludes free-text limit/criteria/institution columns).
#     No hardcoded question numbers. ---
BINARY_ANSWERS = {'yes', 'no', 'sometimes', 'no data', 'not applicable'}

def is_binary_col(series):
    """True if >=80% of non-null answers fall in the binary answer set (case-insensitive, trimmed)."""
    vals = series.dropna().astype(str).str.strip().str.lower()
    if len(vals) == 0:
        return False
    return vals.isin(BINARY_ANSWERS).mean() >= 0.80

binary_cols = [c for c in question_cols if is_binary_col(raw[c])]
print(f"Binary questions detected: {len(binary_cols)} of {len(question_cols)}")

# --- Map answers to numeric: Yes=1, No=0, Sometimes=0.5; No data / Not applicable -> NaN ---
def to_numeric_answer(v):
    if not isinstance(v, str):
        return float('nan')
    s = v.strip().lower()
    if s == 'yes':
        return 1.0
    if s == 'no':
        return 0.0
    if s == 'sometimes':
        return 0.5
    return float('nan')  # 'no data', 'not applicable', other -> NaN

scored = raw[key_cols].copy()
for c in binary_cols:
    scored[c] = raw[c].map(to_numeric_answer)

print("scored frame built with numeric binary columns.")

Raw: (228, 61)
Binary questions detected: 38 of 58
scored frame built with numeric binary columns.


In [10]:
import re

# Build the directionally-defensible subset: only questions where "Yes = better governance"
# is defensible (transparency/disclosure + anti-corruption source bans + state-resource & vote-buying bans).
# This directionality is a documented SUBSTANTIVE JUDGMENT (see framework_decisions.md) — it cannot be
# auto-detected, so the included questions are specified explicitly by (category prefix, question number).
INCLUDED_QUESTIONS = [
    # Category A — anti-corruption source bans
    ('Bans and limits on private income', 1),   # ban foreign donations to parties
    ('Bans and limits on private income', 2),   # ban foreign donations to candidates
    ('Bans and limits on private income', 7),   # ban anonymous donations to parties
    ('Bans and limits on private income', 8),   # ban anonymous donations to candidates
    ('Bans and limits on private income', 9),   # ban govt-contractor donations to parties
    ('Bans and limits on private income', 10),  # ban govt-contractor donations to candidates
    ('Bans and limits on private income', 11),  # ban partial-state-owned-firm donations to parties
    ('Bans and limits on private income', 12),  # ban partial-state-owned-firm donations to candidates
    ('Bans and limits on private income', 13),  # ban abuse of state resources
    ('Bans and limits on private income', 26),  # ban procurement-linked donors
    ('Bans and limits on private income', 27),  # require donations through banking system
    # Category C — unambiguous spending good
    ('Regulations of spending', 38),            # ban vote buying
    # Category D — reporting, oversight, disclosure (all defensible)
    ('Reporting, oversight and sanctions', 47), # parties report regularly
    ('Reporting, oversight and sanctions', 48), # parties report campaign finances
    ('Reporting, oversight and sanctions', 49), # candidates report campaign finances
    ('Reporting, oversight and sanctions', 50), # third parties report
    ('Reporting, oversight and sanctions', 51), # reports made public
    ('Reporting, oversight and sanctions', 52), # reports reveal donor identity
    ('Reporting, oversight and sanctions', 53), # itemized income
    ('Reporting, oversight and sanctions', 54), # itemized spending
]

# Resolve each (prefix, qnum) to its exact column in the scored frame.
# Column format: "<category prefix> - <N>. <question text>"; " - N. " uniquely identifies the question.
def find_question_col(prefix, qnum):
    pattern = f" - {qnum}. "
    matches = [c for c in scored.columns if c.startswith(prefix) and pattern in c]
    if len(matches) != 1:
        raise ValueError(f"Expected 1 column for {prefix} Q{qnum}, found {len(matches)}: {matches}")
    return matches[0]

included_cols = [find_question_col(p, q) for p, q in INCLUDED_QUESTIONS]
print(f"Included questions resolved: {len(included_cols)} (expected {len(INCLUDED_QUESTIONS)})")

# Equal-weighted score = row-wise mean of the included binary questions (skipna -> No data/NA excluded).
out = scored[['Country', 'ISO3']].copy()
out['polfin_transparency_integrity'] = scored[included_cols].mean(axis=1, skipna=True)
out['polfin_n_answered'] = scored[included_cols].notna().sum(axis=1)  # transparency on missingness

# Exclude regional aggregates (e.g. EUR, African Union) — keep rows with a real 3-letter ISO3.
# Regional entities in IDEA's list either lack a standard ISO3 or carry a non-country code; we keep
# only plausible sovereign codes (3 alpha chars) and drop known non-country aggregates explicitly.
out = out[out['ISO3'].notna()].copy()
out['ISO3'] = out['ISO3'].astype(str).str.strip()
out = out[out['ISO3'].str.fullmatch(r'[A-Za-z]{3}')]
out = out.rename(columns={'ISO3': 'country_code'}).drop(columns=['Country'])

# Drop countries with ZERO answered questions — these are outside the measure's coverage (authoritarian
# one-party states with no party-finance regime to code, plus small dependencies in the export shell).
out = out[out['polfin_n_answered'] > 0].copy()

# Reliability floor: a score from very few answered questions isn't trustworthy. Where fewer than
# MIN_ANSWERED (but >0) questions were answered, set the score to NaN but KEEP the row + count, so the
# country stays visible and is honestly flagged as thin-data. 10 = half the questions (reliability
# constant, not a data vintage).
MIN_ANSWERED = 10
out['polfin_transparency_integrity'] = out['polfin_transparency_integrity'].where(
    out['polfin_n_answered'] >= MIN_ANSWERED, other=float('nan'))
out = out.reset_index(drop=True)

n_scored = out['polfin_transparency_integrity'].notna().sum()
n_nan = out['polfin_transparency_integrity'].isna().sum()
print(f"Dropped zero-answer rows. Reliability floor (≥{MIN_ANSWERED}): "
      f"{n_scored} scored, {n_nan} kept as NaN (thin data), {len(out)} total rows")

print(f"\nFinal: {out.shape}, countries: {out['country_code'].nunique()}")
print(f"Score range: {out['polfin_transparency_integrity'].min():.3f} — {out['polfin_transparency_integrity'].max():.3f}")
print(f"Questions answered per country: min {out['polfin_n_answered'].min()}, max {out['polfin_n_answered'].max()}")
print("\nTop 10 (most transparency/oversight provisions):")
print(out.sort_values('polfin_transparency_integrity', ascending=False).head(10).to_string(index=False))
print("\nBottom 10:")
print(out.sort_values('polfin_transparency_integrity').head(10).to_string(index=False))

Included questions resolved: 20 (expected 20)
Dropped zero-answer rows. Reliability floor (≥10): 177 scored, 3 kept as NaN (thin data), 180 total rows

Final: (180, 3), countries: 180
Score range: 0.050 — 1.000
Questions answered per country: min 2, max 20

Top 10 (most transparency/oversight provisions):
country_code  polfin_transparency_integrity  polfin_n_answered
         LBR                       1.000000                 18
         LVA                       1.000000                 15
         PRT                       1.000000                 16
         MDA                       0.973684                 19
         SRB                       0.973684                 19
         USA                       0.973684                 19
         MNE                       0.950000                 20
         BRA                       0.950000                 20
         CAN                       0.944444                 18
         EST                       0.944444                 18


In [11]:
# Data currency: this source is a wave-updated cross-section; the export carries no data-year column,
# so we stamp the retrieval date and note the vintage is wave-based (latest IDEA update round) below.
retrieval_date = datetime.today().strftime("%Y-%m-%d")

output_path = os.path.join(PROCESSED_DIR, "polfinance_clean.csv")
out.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {out.shape}")

n_countries = out['country_code'].nunique()

update_entry(
    "TI_POLFINANCE",
    last_successful_download_date=retrieval_date,
    data_as_of_date=retrieval_date,   # retrieval date — see note; source has no data-year column
    local_filename="polfinance_clean.csv",
    latest_available_version="wave-updated cross-section",
    notes=("Political-finance TRANSPARENCY & oversight score (polfin_transparency_integrity, 0-1). "
           "Source: International IDEA Political Finance Database (NOT Transparency International — TI does "
           "report-based analysis; IDEA holds the structured 181-country data). Automated .xlsx export. "
           "DE JURE regulation only — measures rules on paper, NOT enforcement/compliance/actual money. "
           "Equal-weighted mean of 20 directionally-defensible binary questions (transparency/disclosure + "
           "anti-corruption source bans + state-resource & vote-buying bans); contested questions "
           "(contribution/spending limits, public funding, corporate/union bans) EXCLUDED — see framework_decisions.md. "
           "Reliability floor: <10 of 20 answered -> NaN (3 countries). polfin_n_answered carries count. "
           "Wave-updated cross-section (no data-year in export); vintage = latest IDEA update round. "
           f"Coverage: {n_countries} countries.")
)
print_entry("TI_POLFINANCE")

Written: C:\Users\mjbou\governance-framework\data\processed\polfinance_clean.csv
Shape: (180, 3)
[download_log] Updated entry for TI_POLFINANCE
  source_id: TI_POLFINANCE
  last_attempted_date: 2026-06-17
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2026-06-17
  local_filename: polfinance_clean.csv
  latest_available_version: wave-updated cross-section
  no_update_reason: nan
  notes: Political-finance TRANSPARENCY & oversight score (polfin_transparency_integrity, 0-1). Source: International IDEA Political Finance Database (NOT Transparency International — TI does report-based analysis; IDEA holds the structured 181-country data). Automated .xlsx export. DE JURE regulation only — measures rules on paper, NOT enforcement/compliance/actual money. Equal-weighted mean of 20 directionally-defensible binary questions (transparency/disclosure + anti-corruption source bans + state-resource & vote-buying bans); contested questions (contribution/spending limits, public funding,